
In this project, I attempt to develop a workflow that uniquely identify pathogens within a metagenomic sample. I compare the genome of Escherichia coli O157:H7 (a pathogenic strain) with SE11 (a non-pathogenic strain). These strains, while closely related (sharing a large portion of their genomes since they belong to the same species), have key genomic differences that contribute to their pathogenicity and ecological niches. O157:H7 has additional pathogenicity islands, virulence genes, and mobile genetic elements (e.g., Shiga toxin genes, plasmids) that distinguish it as a pathogen. unlike 0157:H7, SE11 being non-pathogenic, lacks these virulence factors but share homologous sequences. Identifying these regions is crucial for designing specific markers for metagenomic detection. Plasmids, insertion sequences, and prophages in O157:H7 are key targets. These are often strain-specific and can be used to differentiate pathogenic strains from non-pathogenic ones.




# **Goals** #

- Identify and extract divergent regions (Regions that exists between alignment block/coordinates in the refrence)
- Validate selection specificity using BLAST 
- Test Specificity in Metagenomic Context



# 1. **Introduction** #
     **Why MUMmer**
MUMmer is an open source software package for the rapid alignment of very large DNA and amino acid sequences. MUMmer is a suffix tree algorithm specifically designed to find all maximal exact matches (MEMs) of a specified minimum length between reference and query sequences. The algorithm is ideal for handling large datasets due to their ability to be constructed and searched in linear time and space. This allows MUMmer to quickly identify maximal exact matches between large genomes, making it highly efficient for genomic analysis.


   **Input Files**

files were obatained from the NCBI public repository.  omplete genomes were used.
Path to ref sequence : /Users/user/Documents/MISSION/Data_analytics/Bioinformatics/ucd_project/ucd_project2/ref_0157H7.fasta
Path to query Sequence : /Users/user/Documents/MISSION/Data_analytics/Bioinformatics/ucd_project/ucd_project2/qry_SE11.fasta

### **Activate Conda Environment** ###

In [ ]:
! Conda env list 

### **Import Dependencies** ###

In [ ]:

import os 
import pandas as pd
import subprocess 
import matplotlib.pyplot as plt
import seaborn


In [ ]:
ref = "data/ref_0157H7.fasta"
qry = "data/qry_SE11.fasta"

In [ ]:
! mummer -maxmatch -l 30 ref_0157H7.fasta qry_SE11.fasta> matches_mum.txt

The code above generate a .mum file that contains all raw matches. The output has now been forced to .txt for more convenient parsing.
 The -l flag specifies the minimum match length of unique match. while the -maxmatch finds all maximum exact matches.This is crucial to filter out noise, such as very short matches that are less likely to have biological significance. A longer minimum match length reduces false positives. For metagenomic identification, It is to consider a length that balances specificity (to distinguish the pathogen) and sensitivity (to ensure detection). 




# **Workflow Using Nucmer** #

Compared to MUMmer, NUCmer aligns larger alignment blocks allowing mismatches and gaps between genomes. Gaps in the alignment correspond to divergent regions.

### **Run Alignment with NUCmer** ###

In [ ]:
! nucmer --prefix=nucmer_alignment ref_0157H7.fasta qry_SE11.fasta

This next command will list the coordinates, percent identities and other useful statistics of each alignment in a table. Each line of the table represents an individual pairwise alignment, and each line is sorted by its starting reference coordinate (-r). Additional information, like alignment coverage (-c) and sequence length (-l) can be added to the table with the appropriate options.

The resulting .delta output is an encoded file that represents the alignment between the two inputs, however to view the output, it is necessary to parse the nucmer.delta file with the show-coords utility in order to extract useful information from the comparison. 

 To make the .coords output from show-coords even more appealing, we will it parse it with pandas to enable any daownstream exploration and insight derivation.

In [ ]:
! show-coords -rclT nucmer_alignment.delta > nucmer_alignment.coords

#### **Parse .coords output to Pandas** ###
For easy access and manipulation, lets parse the .coords output to pandas excliuding the header 

In [ ]:

# Path to the .coords file
coords_file = "nucmer_alignment.coords"

# Load the .coords file using row 4 as column headers
coords_df = pd.read_csv(coords_file, sep='\s+', skiprows=4, header=None)

# Relabel columns with meaningful names
coords_df.columns = [
    "Ref_Start",     # Start position in reference genome (S1)
    "Ref_End",       # End position in reference genome (E1)
    "Qry_Start",     # Start position in query genome (S2)
    "Qry_End",       # End position in query genome (E2)
    "Ref_Align_Len", # Alignment length in reference genome (LEN,1)
    "Qry_Align_Len", # Alignment length in query genome (LEN.1,2)
    "Percent_Identity", # Percentage identity (%IDY)
    "Ref_Len",       # Total length of reference genome (LEN.2,R)
    "Qry_Len",       # Total length of query genome (LEN.3,Q)
    "Ref_Coverage",  # Coverage in reference genome (COV,R]
    "Qry_Coverage",  # Coverage in query genome (COV,Q]
    "Ref_Name",      # Reference sequence name (TAGS)
    "Qry_Name"       # Query sequence name (TAGS)
]

# Export DataFrame to a CSV file
coords_df.to_csv("cleaned_nucmer_coords.csv", index=True)


# Display the dataframe
coords_df


From the result, there are 587 regions of alignments between the two genomes.

In [ ]:
! mummerplot -l nucmer_alignment.delta

In [ ]:
file = 'cleaned_nucmer_coords.csv' 
alignment_data = pd.read_csv(file)  # Load the file

alignment_data.columns

In [ ]:

# Plot the Percent Identity Distribution
plt.figure(figsize=(8, 4))
plt.hist(alignment_data['Percent_Identity'], bins=20, color='skyblue', edgecolor='black')
plt.title('Percent Identity Distribution', fontsize=14)
plt.xlabel('Percent Identity', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Plot the Alignment Length Distribution
plt.figure(figsize=(8, 4))
plt.hist(alignment_data['Ref_Align_Len'], bins=30, color='salmon', edgecolor='black')
plt.title('Alignment Length Distribution in Ref', fontsize=14)
plt.xlabel('Alignment Length (bp)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### **Filter for reference-specific alignments / Remove Overlaps** ###

For this project, focused on regions unique to O157:H7, removing overlaps and repeating is essential to ensure accurate and non-redundant alignments. Overlaps can artificially inflate the number of matches, complicating downstream analyses such as divergence calculation or uniqueness assessment. By eliminating overlaps, we simplify interpretation and improve the detection of truly divergent regions, as overlaps can obscure these insights. Additionally, in applications like metagenome read recruitment, overlapping matches introduce ambiguity, while non-overlapping matches provide greater precision and specificity for identifying distinct regions. 

The flags:

-r: Retains matches specific to the reference genome (O157:H7),i.e only the best matches for the reference is retained 
-u: The filtering process retains only unique alignments where each region in the reference genome aligns to a single region in the query genome and vice versa, effectively eliminating ambiguous matches caused by repetitive sequences or multiple alignments.
-i: Filter alignments based on identity match


In [ ]:

! delta-filter -i 95 -m -o 0 nucmer_alignment.delta > nuc_alignment_ref_filtered.delta

### Visualize or Export Alignments: ###  

Again, we employ show-coords utility to view a tabular summary of alignment details which can also be parsed using pandas for downstream manipulations to extract useful insights.
The flags 
-r: Sorts alignments by reference position, -c: Displays coverage details and -l: Displays alignment lengths and -T displays a tabular output format compared to the rather difficult to parse output.

In [ ]:
! show-coords -rclT nuc_alignment_ref_filtered.delta > nuc_alignment_ref_filt.coords


In [ ]:
# Path to the .coords file
coords_file1 = "nuc_alignment_ref_filt.coords"

# Load the .coords file, skipping 4 rows and ensuring flexible delimiters
coords_df1 = pd.read_csv(coords_file1, sep='\s+', skiprows=4, header=None)

# Relabel columns with meaningful names
coords_df1.columns = [
    "Ref_Start",     # Start position in reference genome (S1)
    "Ref_End",       # End position in reference genome (E1)
    "Qry_Start",     # Start position in query genome (S2)
    "Qry_End",       # End position in query genome (E2)
    "Ref_Align_Len", # Alignment length in reference genome (LEN,1)
    "Qry_Align_Len", # Alignment length in query genome (LEN.1,2)
    "Percent_Identity", # Percentage identity (%IDY)
    "Ref_Len",       # Total length of reference genome (LEN.2,R)
    "Qry_Len",       # Total length of query genome (LEN.3,Q)
    "Ref_Coverage",  # Coverage in reference genome (COV,R]
    "Qry_Coverage",  # Coverage in query genome (COV,Q]
    "Ref_Name",      # Reference sequence name (TAGS)
    "Qry_Name"       # Query sequence name (TAGS)
]

# Export DataFrame to a CSV file
coords_df1.to_csv("cleaned_ref_filt_coords.csv", index=True)


# Display the dataframe
coords_df1.tail(20)

### **Call Divergent Regions in reference** ###

The show-diff command compares the filtered alignments (shared regions) against the entire reference genome. It explicitly identifies divergent regions, which are the regions in the reference genome (O157:H7) that are not aligned to the query genome (SE11). These divergent regions are essentially the parts of the reference sequence that remain unaligned after running NUCmer and filtering for matches.

In [ ]:
! show-diff nuc_alignment_ref_filtered.delta > nuc_ref_divergent_regions.delta


4. ### **Identify Unaligned Regions Using Bedtools**  ###
#### **Generate .bed files for both genomes from the filtered, alignment coordinate file** ####
step 1: Generate a BED file of aligned regions from .coords:
This next sets of command processes the .coords file, extracting the alignment positions (start and end) and writing them into a BED file format (aligned_regions.bed). The $1-1 is used to convert the 1-based start position to the 0-based start position, which is the standard for BED format.

In [ ]:
#! awk -F, '{print $13, $2-1, $3, $6}' cleaned_ref_filt_coords.csv > aligned_ref_0157H7.bed
#! awk -F, '{print $14, $4-1, $5, $7}' cleaned_ref_filt_coords.csv > aligned_qry_SE11.bed



! awk 'BEGIN {OFS="\t"} /^[0-9]/ {print $12, $1-1, $2, $5}' nuc_alignment_ref_filt.coords > aligned_ref.bed
! awk 'BEGIN {OFS="\t"} /^[0-9]/ {print $13, $3-1, $4, $6}' nuc_alignment_ref_filt.coords > aligned_qry.bed

## **Step 3: Use bedtools complement to identify unaligned regions** ###
### **Extract Divergent regions** ###
Step 2: To compute complement for the reference genome (ref_0157H7) using bedtools complement, we  need to first create the index file. The .fai file contains information about the lengths of the sequences in the reference genome and is required by many tools in genomic analyses.

Run the following command to generate the .fai file:

In [ ]:
! samtools faidx ref_0157H7.fasta

#### **Compute complement for aligned.bed file for ref_0157H7** ####
This will produce a BED file (divergent_regions.bed) that contains the regions in the reference genome that are not aligned in the .coords file. These regions are "divergent" in the sense that they don't match the aligned regions, which is what we are interested in for the project.

In [ ]:
! bedtools complement -i aligned_ref.bed -g ref_0157H7.fasta.fai > ref_divergent_regions.bed


In [ ]:
! bedtools getfasta -fi ref_0157H7.fasta -bed ref_divergent_regions.bed -fo ref_divergent_sequences.fasta


From the result, there are 223 divergent regions/sequences in the reference genome.

O157:H7 has a larger genome (~5.5 Mb) compared to SE11 (~4.8 Mb), largely due to its accessory genome. This makes exact match identification feasible.


In [ ]:
! conda install -C blast 